# Fondo Auto — Piano a scalini (cash + 3 ETF) e scenari

Versamento **€210/mese × 14** diviso in: conto deposito + 3 ETF a scadenza diversa.
Alla **prima scadenza** il capitale viene reinvestito in un altro ETF; alle **altre scadenze** va sul conto deposito.
Poi si fanno variare gli **anni di scadenza di ±1** e il **tasso del conto deposito (1–2%)** per vedere i panorami possibili.

Dati: `rendimenti_etf.xlsx` (ETF **Acc**, **Dist** o entrambi: vedi `PROVENTI`; i Dist sono modellati come se reinvestissero la cedola al netto dello stesso rendimento), rendimento = colonna **Netto TER e tasse**.
Metriche: *valore finale*, *XIRR* (rendimento annuo dei versamenti), *perdita se tassi +1%* (≈ valore ancora investito × anni residui × 1%).

In [21]:
import itertools
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import brentq

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ---- Parametri ----
HORIZON_YEARS = 8               # tra quanti anni serve l'auto
PROVENTI = ("Acc", "Dist")    # strategia: ("Acc",) accumulazione, ("Dist",) distribuzione, ("Acc", "Dist") entrambe
XLSX = "../data/rendimenti_etf.xlsx"
MONTHLY_PAYMENT = 210.0
EXTRA_MONTHS = (6, 12)          # mesi con 2ª mensilità -> 14 versamenti/anno
START_DATE = "2026-09-01"
TARGET = 35_000                 # costo auto
CAR_SALE = 4_000                # valore stimato auto attuale
CASH_RETURN = 0.015             # tasso conto deposito di base (range 1-2%)
STRESS = 0.01                   # shock tassi per la colonna rischio

## 1. Universo: ETF secondo `PROVENTI`

In [22]:
raw = pd.read_excel(XLSX, header=3)
print(pd.read_excel(XLSX, header=None).iloc[2, 0])

# Le formule dell'xlsx non hanno valori in cache (file scritto da script): ricalcolo qui, stessa formula di Excel
tax = pd.read_excel(XLSX, header=None, nrows=2).iloc[:, 1]      # B1 aliquota Govt, B2 aliquota Corp
raw["Netto TER e tasse"] = (raw["Rendimento lordo"] - raw["TER"]) * (1 - raw["Tipo"].map(lambda t: tax[0] if t == "Govt" else tax[1]))

universe = raw[raw["Proventi"].isin(PROVENTI)].copy()
universe["Short"] = universe["Emittente"] + " " + universe["Scadenza"].dt.year.astype(str) + " " + universe["Proventi"]
universe = universe.set_index("ISIN")
print(f"{len(universe)} ETF {"/".join(PROVENTI)} su {len(raw)} totali")
universe[["Short", "Nome", "Tipo", "Scadenza", "Rendimento lordo", "TER", "Netto TER e tasse"]] \
    .sort_values("Scadenza")

Aggiornato il 05/10/2026 15:08 da yield_etf.xlsx
54 ETF Acc/Dist su 54 totali


,Short,Nome,Tipo,Scadenza,Rendimento lordo,TER,Netto TER e tasse
ISIN,,,,,,,
IE000SIZJ2B2,iShares 2026 Dist,iShares iBonds Dec 2026 Term € Corp UCITS ETF,Corp,2026-12-31,0.0278,0.0012,0.0197
IE000AYJ75E5,Invesco 2026 Dist,Invesco BulletShares 2026 EUR Corporate Bond U...,Corp,2026-12-31,0.0294,0.0010,0.0210
IE0004QWOTD5,Invesco 2026 Acc,Invesco BulletShares 2026 EUR Corporate Bond U...,Corp,2026-12-31,0.0294,0.0010,0.0210
LU2872291278,Amundi 2026 Acc,Amundi Fixed Maturity 2026 Euro Government Bon...,Govt,2026-12-31,0.0263,0.0009,0.0222
IE000LZ7BZW8,iShares 2026 Dist,iShares iBonds Dec 2026 Term € Italy Govt Bond...,Govt,2026-12-31,0.0270,0.0012,0.0226
LU2673522830,DWS 2027 Acc,Xtrackers II Target Maturity Sept 2027 EUR Cor...,Corp,2027-09-30,0.0257,0.0012,0.0181
LU2673523218,DWS 2027 Dist,Xtrackers II Target Maturity Sept 2027 EUR Cor...,Corp,2027-09-30,0.0257,0.0012,0.0181
IE000XOS4OJ6,Invesco 2027 Dist,Invesco BulletShares 2027 EUR Corporate Bond U...,Corp,2027-12-31,0.0331,0.0010,0.0238
LU2780871823,Amundi 2027 Dist,Amundi Fixed Maturity 2027 German Bund Governm...,Govt,2027-12-31,0.0293,0.0009,0.0248


## 2. Calendario dei versamenti
12 versamenti ordinari + 2 extra (giugno e dicembre) = 14 mensilità/anno. La lunghezza segue `HORIZON_YEARS`.

In [23]:
def create_cashflow(years, start=START_DATE, pay=MONTHLY_PAYMENT, extra_months=EXTRA_MONTHS):
    d = pd.date_range(start, periods=years * 12, freq="ME")
    return pd.DataFrame({"date": d, "contribution": [pay * (2 if m in extra_months else 1) for m in d.month]})


cashflow = create_cashflow(HORIZON_YEARS)
calendar = (cashflow.assign(year=cashflow.date.dt.year, month=cashflow.date.dt.month)
            .pivot(index="year", columns="month", values="contribution").fillna(0))
calendar["Totale"] = calendar.sum(axis=1)
print(f"Orizzonte: {HORIZON_YEARS} anni, acquisto auto a {cashflow.date.iloc[-1]:%m/%Y}, "
      f"versato €{cashflow.contribution.sum():,.0f}")
calendar.style.format("{:,.0f}")

Orizzonte: 8 anni, acquisto auto a 08/2034, versato €23,520


month,1,2,3,4,5,6,7,8,9,10,11,12,Totale
year,,,,,,,,,,,,,
2026,0,0,0,0,0,0,0,0,210,210,210,420,"1,050"
2027,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2028,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2029,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2030,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2031,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2032,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2033,210,210,210,210,210,420,210,210,210,210,210,420,"2,940"
2034,210,210,210,210,210,420,210,210,0,0,0,0,"1,890"


## 3. Simulatore
Mese per mese: scadenza ETF → capitale su monetario → versamento ripartito per peso → rendimento mensile.

In [24]:
def simulate(cf, weights, dest=None, cash_rate=CASH_RETURN, rate_shift=0.0):
    """weights: {asset: quota del versamento}, asset = ISIN o "CASH".
    dest: {ISIN: asset} dove va il capitale quando quell'ETF scade (default: CASH)."""
    dest = dest or {}
    isins = {*weights, *dest.values()} - {"CASH"}
    m = lambda a: (1 + a) ** (1 / 12) - 1
    rate = {i: m(universe.at[i, "Netto TER e tasse"] + rate_shift) for i in isins}
    mat = {i: universe.at[i, "Scadenza"] for i in isins}
    val = {**{i: 0.0 for i in isins}, "CASH": 0.0}
    tot_w = sum(weights.values())

    def route(a, now):          # segue la catena finché l'ETF di destinazione è scaduto
        while a != "CASH" and now >= mat[a]:
            a = dest.get(a, "CASH")
        return a

    rows = []
    for _, r in cf.iterrows():
        for i in isins:
            if val[i] and r.date >= mat[i]:
                val[route(i, r.date)] += val[i]; val[i] = 0.0
        for a, w in weights.items():
            val[route(a, r.date)] += r.contribution * w / tot_w
        for i in isins:
            val[i] *= 1 + rate[i]
        val["CASH"] *= 1 + m(cash_rate)
        rows.append({"date": r.date, "contribution": r.contribution, **val, "total": sum(val.values())})
    return pd.DataFrame(rows)


def xirr(sim):
    t0 = sim.date.iloc[0]
    cfs = (-sim.contribution).tolist(); cfs[-1] += sim.total.iloc[-1]
    yrs = [(d - t0).days / 365.25 for d in sim.date]
    return brentq(lambda r: sum(c / (1 + r) ** y for c, y in zip(cfs, yrs)), -0.9, 1)


def stress_loss(sim):
    # ponytail: duration ~ anni residui a scadenza (ok per bullet, no convexity)
    end, last = sim.date.iloc[-1], sim.iloc[-1]
    etfs = [c for c in sim.columns if c not in ("date", "contribution", "CASH", "total")]
    return sum(last[i] * max(0, (universe.at[i, "Scadenza"] - end).days / 365.25) * STRESS for i in etfs)

## 4. Il piano base

In [25]:
# ---- Il piano ----
CASH_SLICE = 30                              # €/mese sul conto deposito
PLAN_YEARS = {2029: 50, 2031: 60, 2033: 70}  # anno di scadenza -> €/mese
REINVEST_YEAR = 2032                         # ETF dove va il capitale della PRIMA scadenza (le altre -> conto deposito)
CASH_RATES = (0.01, 0.015, 0.02)


def best_etf(year):
    s = universe[universe.Scadenza.dt.year == year]
    return s["Netto TER e tasse"].idxmax() if len(s) else None   # a parità di anno, il migliore netto


def build_plan(years):
    """years: {anno: €/mese} -> (weights, dest) oppure None se manca un ETF per quell'anno."""
    ids = {y: best_etf(y) for y in years}
    rein = best_etf(REINVEST_YEAR)
    if rein is None or None in ids.values():
        return None
    weights = {"CASH": CASH_SLICE}
    for y, amt in years.items():
        weights[ids[y]] = weights.get(ids[y], 0) + amt
    first = min(weights.keys() - {"CASH"}, key=lambda i: universe.at[i, "Scadenza"])
    return weights, {first: rein}


def evaluate(years, cash_rate):
    plan = build_plan(years)
    if plan is None:
        return None
    s = simulate(cashflow, *plan, cash_rate=cash_rate)
    fin = s.total.iloc[-1]
    return s, {"Valore finale": fin, "XIRR": xirr(s), "Perdita se tassi +1%": stress_loss(s),
               "Gap con vendita auto": TARGET - CAR_SALE - fin, "% in monetario": s.CASH.iloc[-1] / fin}


w, d = build_plan(PLAN_YEARS)
rein = universe.at[best_etf(REINVEST_YEAR), "Short"]
pd.DataFrame([{"ETF": universe.at[i, "Short"] if i != "CASH" else "Conto deposito",
               "Scadenza": universe.at[i, "Scadenza"].date() if i != "CASH" else None,
               "Netto": universe.at[i, "Netto TER e tasse"] if i != "CASH" else CASH_RETURN,
               "€/mese": amt,
               "Alla scadenza": "" if i == "CASH" else (f"reinvestito in {rein}" if i in d else "conto deposito")}
              for i, amt in w.items()])

,ETF,Scadenza,Netto,€/mese,Alla scadenza
0,Conto deposito,None,0.0150,30,
1,iShares 2029 Dist,2029-12-31,0.0357,50,reinvestito in DWS 2032 Dist
2,iShares 2031 Dist,2031-12-31,0.0305,60,conto deposito
3,iShares 2033 Dist,2033-12-31,0.0319,70,conto deposito


In [26]:
sim, base = evaluate(PLAN_YEARS, CASH_RETURN)
inv = sim.contribution.sum()
labels = {**{i: universe.at[i, "Short"] for i in sim.columns if i in universe.index}, "CASH": "Conto deposito"}
pd.Series({"Versato": inv, "Valore finale": base["Valore finale"], "Guadagno": base["Valore finale"] - inv,
           "XIRR annuo": base["XIRR"], "Gap obiettivo": TARGET - base["Valore finale"],
           "Gap con vendita auto": base["Gap con vendita auto"]}).to_frame("Piano base")

,Piano base
Versato,"23,520.0000"
Valore finale,"25,928.3152"
Guadagno,"2,408.3152"
XIRR annuo,0.0246
Gap obiettivo,"9,071.6848"
Gap con vendita auto,"5,071.6848"


In [27]:
fig = go.Figure()
fig.add_scatter(x=sim.date, y=sim.total, name="Valore portafoglio", line_width=3)
fig.add_scatter(x=sim.date, y=sim.contribution.cumsum(), name="Capitale versato", line_dash="dash")
fig.add_hline(y=TARGET, line_dash="dot", line_color="red", annotation_text="Obiettivo auto")
fig.update_layout(title="Valore vs capitale versato", yaxis_title="€")
fig.show()

fig = go.Figure()
for col in labels:
    fig.add_scatter(x=sim.date, y=sim[col], stackgroup="one", name=labels[col])
fig.update_layout(title="Composizione nel tempo", yaxis_title="€")
fig.show()

## 5. Panorami: scadenze ±1 anno × tasso conto deposito
Ogni scadenza del piano (2029 / 2031 / 2033) si sposta di −1, 0, +1 anno: 27 combinazioni.
Escluse quelle con **due ETF nella stessa scadenza** o con **anno senza ETF** (es. 2033+1 = 2034): l'elenco è sotto.
Tasso del conto deposito all'1%, 1,5%, 2%.

In [28]:
rows, skipped = [], []
for shifts in itertools.product((-1, 0, 1), repeat=len(PLAN_YEARS)):
    ys = [y + sh for y, sh in zip(PLAN_YEARS, shifts)]
    label = "/".join(map(str, ys))
    if len(set(ys)) < len(ys):
        skipped.append((label, "due ETF con la stessa scadenza")); continue
    if missing := [y for y in ys if best_etf(y) is None]:
        skipped.append((label, f"nessun ETF con scadenza {missing[0]}")); continue
    for cr in CASH_RATES:
        rows.append({"Scadenze": label, "Conto deposito": cr, **evaluate(dict(zip(ys, PLAN_YEARS.values())), cr)[1]})
scen = pd.DataFrame(rows)
print(f"{scen.Scadenze.nunique()} combinazioni valide su {3 ** len(PLAN_YEARS)}; escluse {len(skipped)}:")
display(pd.DataFrame(skipped, columns=["Scadenze", "Motivo"]).set_index("Scadenze"))

# valore finale per scadenze x tasso deposito
piv = scen.pivot_table(index="Scadenze", columns="Conto deposito", values="Valore finale").sort_values(CASH_RETURN, ascending=False)
piv.style.format("{:,.0f}")

21 combinazioni valide su 27; escluse 6:


,Motivo
Scadenze,
2028/2032/2032,due ETF con la stessa scadenza
2029/2032/2032,due ETF con la stessa scadenza
2030/2030/2032,due ETF con la stessa scadenza
2030/2030/2033,due ETF con la stessa scadenza
2030/2030/2034,due ETF con la stessa scadenza
2030/2032/2032,due ETF con la stessa scadenza


Conto deposito,0.010000,0.015000,0.020000
Scadenze,,,
2029/2032/2034,"25,953","26,144","26,337"
2028/2032/2034,"25,935","26,125","26,318"
2030/2032/2034,"25,933","26,124","26,317"
2029/2031/2034,"25,857","26,066","26,278"
2028/2031/2034,"25,838","26,048","26,260"
2030/2031/2034,"25,837","26,046","26,258"
2029/2032/2033,"25,784","26,006","26,230"
2029/2030/2034,"25,774","26,004","26,238"
2028/2032/2033,"25,766","25,987","26,211"


In [29]:
mid = scen[scen["Conto deposito"] == CASH_RETURN].sort_values("Valore finale", ascending=False)
mid.drop(columns="Conto deposito").reset_index(drop=True)

,Scadenze,Valore finale,XIRR,Perdita se tassi +1%,Gap con vendita auto,% in monetario
0,2029/2032/2034,"26,144.0767",0.0266,7.3550,"4,855.9233",0.6575
1,2028/2032/2034,"26,125.4924",0.0265,7.3550,"4,874.5076",0.6572
2,2030/2032/2034,"26,123.9192",0.0264,7.3550,"4,876.0808",0.6572
3,2029/2031/2034,"26,066.3864",0.0259,7.3550,"4,933.6136",0.6565
4,2028/2031/2034,"26,047.8021",0.0257,7.3550,"4,952.1979",0.6562
5,2030/2031/2034,"26,046.2289",0.0257,7.3550,"4,953.7711",0.6562
6,2029/2032/2033,"26,006.0055",0.0253,0.0000,"4,993.9945",1.0000
7,2029/2030/2034,"26,004.1375",0.0253,7.3550,"4,995.8625",0.6556
8,2028/2032/2033,"25,987.4212",0.0251,0.0000,"5,012.5788",1.0000
9,2030/2032/2033,"25,985.8480",0.0251,0.0000,"5,014.1520",1.0000


In [30]:
fig = go.Figure(go.Bar(
    y=mid["Scadenze"], x=mid["Valore finale"], orientation="h", text=mid["XIRR"].map("{:.2%}".format),
    marker=dict(color=mid["XIRR"], colorscale="Viridis")))
fig.add_vline(x=TARGET - CAR_SALE, line_dash="dot", line_color="orange", annotation_text="Obiettivo − vendita auto")
fig.update_layout(title=f"Valore finale per scadenze — conto deposito {CASH_RETURN:.1%} (etichetta = XIRR)",
                  xaxis_title="Valore finale (€)", yaxis=dict(autorange="reversed"), height=500)
fig.show()